# 🔧 LoRA 파인튜닝

## LoRA (Low-Rank Adaptation)란?

LoRA는 대형 언어 모델(LLM)을 효율적으로 미세조정하는 기법입니다.

### 핵심 원리
- 사전 학습된 가중치 행렬 **W**에 저차원(low-rank) 분해 행렬 **ΔW = BA**를 더합니다
- 원래 가중치 W는 고정(freeze)하고, A와 B만 학습합니다
- 전체 파라미터의 **1% 미만**만 학습하면서도 우수한 성능을 달성합니다

### 왜 LoRA를 사용하나?
| 장점 | 설명 |
|------|------|
| **메모리 효율** | 전체 모델 대비 훨씬 적은 GPU 메모리 사용 |
| **빠른 학습** | 학습 파라미터가 적어 수렴이 빠름 |
| **모듈성** | 어댑터만 교체하여 다양한 도메인 적용 가능 |
| **원본 보존** | 기본 모델 가중치를 변경하지 않음 |

### 이 노트북의 설정
- **기본 모델**: `Qwen/Qwen3-4B-Instruct-2507`
- **LoRA rank (r)**: 16, **alpha**: 32
- **대상 모듈**: attention (Q/K/V/O) + MLP (gate/up/down)
- **손실 마스킹**: assistant 응답만 학습 (시스템/사용자 메시지는 제외)

> ⚠️ 이 노트북은 **준비된 데이터 번들**을 사용합니다.  
> SDG, 교사 모델 호출, 데이터 수정은 하지 않습니다.  
> 번들이 유효하지 않으면 학습이 중단됩니다.

In [ ]:
"""환경 부트스트랩 — local과 workbench 모두 지원."""

import subprocess, sys
from pathlib import Path

# 프로젝트 루트 탐색 (노트북 위치 기준)
_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

# 패키지 설치 확인 및 자동 설치
try:
    import rhoai_model_training_lab  # noqa: F401
    print("✅ rhoai_model_training_lab 패키지 확인됨")
except ImportError:
    print("📦 패키지 설치 중... (최초 1회)")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-e", str(_project_root)],
        stdout=subprocess.DEVNULL,
    )
    print("✅ 설치 완료 — 커널 재시작이 필요할 수 있습니다.")


In [ ]:
"""Load configuration and validate bundle compatibility."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_training_config, load_bundle_config, PROJECT_ROOT,
)
from rhoai_model_training_lab.data import BundleManager

load_env()

# Load LoRA config
lora_config = load_training_config("lora")
model_id = lora_config["model"]["model_id"]
model_revision = lora_config["model"]["model_revision"]

print(f"모델: {model_id} (rev: {model_revision})")
print(f"LoRA r={lora_config['lora']['r']}, alpha={lora_config['lora']['lora_alpha']}")
print(f"대상 모듈: {lora_config['lora']['target_modules']}")
print(f"시드: {lora_config['training']['seed']}")

# Load and validate bundle
release_config = load_bundle_config()
bundle_base = release_config.get("bundle", {}).get("base_path", "data/prepared/tau-knowledge-v1")
bundle_path = PROJECT_ROOT / bundle_base

print(f"\n번들 경로: {bundle_path}")
mgr = BundleManager.load_bundle(bundle_path)
manifest = mgr.manifest

# Compatibility check
compat = mgr.validate_compatibility(model_id)
if compat.errors:
    for err in compat.errors:
        print(f"  ❌ {err}")
    raise RuntimeError(
        "번들 호환성 검증 실패 — 학습을 진행할 수 없습니다.\n"
        "data_preparation/ 노트북에서 올바른 번들을 생성하세요."
    )

print(f"\n✅ 번들 호환성 검증 통과")
print(f"   학습 샘플: {manifest.canonical_train_count}")
print(f"   검증 샘플: {manifest.canonical_validation_count}")
print(f"   번들 버전: {manifest.bundle_version}")

In [ ]:
"""Preview training data with loss mask visualization."""

from transformers import AutoTokenizer

# Load tokenizer for visualization
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Load backend-specific training data
train_data = mgr.get_training_samples("lora", "train")
val_data = mgr.get_training_samples("lora", "validation")

print(f"LoRA 학습 데이터: {len(train_data)} 샘플")
print(f"LoRA 검증 데이터: {len(val_data)} 샘플")

# Preview first 2 examples with loss mask visualization
print("\n" + "=" * 70)
print("📝 학습 데이터 미리보기 (손실 마스크 시각화)")
print("=" * 70)

for i, sample in enumerate(train_data[:2]):
    messages = sample.get("messages", [])
    print(f"\n--- 샘플 {i+1} ---")
    print(f"메시지 수: {len(messages)}")

    for msg in messages:
        role = msg["role"]
        content = msg.get("content", "") or ""
        # Loss mask: only assistant messages contribute to loss
        is_target = role == "assistant"
        mask_icon = "🎯 [학습 대상]" if is_target else "🚫 [마스킹됨]"

        display_content = content[:200] + "..." if len(content) > 200 else content
        print(f"\n  {mask_icon} [{role}]:")
        print(f"    {display_content}")

        if msg.get("tool_calls"):
            print(f"    🔧 도구 호출: {len(msg['tool_calls'])}건")
            for tc in msg["tool_calls"][:2]:
                fn = tc.get("function", {})
                print(f"       → {fn.get('name', '?')}({fn.get('arguments', '')[:80]})")

    # Token count
    try:
        formatted = tokenizer.apply_chat_template(messages, tokenize=True)
        print(f"\n  토큰 수: {len(formatted)}")
        max_len = lora_config["data"]["max_seq_length"]
        if len(formatted) > max_len:
            print(f"  ⚠️  최대 길이 {max_len} 초과!")
    except Exception:
        pass

print(f"\n{'=' * 70}")
print("손실 마스킹 정책: assistant 응답만 학습 대상")
print("시스템 메시지, 사용자 입력, 도구 관찰값은 손실 계산에서 제외됩니다.")

In [ ]:
"""Train using training_hub.lora_sft — actual API call."""

import time
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU가 감지되지 않았습니다.\n"
        "LoRA 학습에는 CUDA 호환 GPU가 필요합니다.\n"
        "GPU가 있는 환경에서 이 노트북을 실행하세요."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB")
print()

# Prepare training_hub arguments from config
train_file = str(PROJECT_ROOT / lora_config["data"]["train_file"])
val_file = str(PROJECT_ROOT / lora_config["data"]["validation_file"])
output_dir = str(PROJECT_ROOT / lora_config["training_args"]["output_dir"])

print("=" * 70)
print("🚀 LoRA 학습 시작")
print("=" * 70)
print(f"  학습 데이터: {train_file}")
print(f"  검증 데이터: {val_file}")
print(f"  출력 디렉토리: {output_dir}")
print(f"  에포크: {lora_config['training_args']['num_train_epochs']}")
print(f"  배치 크기: {lora_config['training_args']['per_device_train_batch_size']}")
print(f"  기울기 누적: {lora_config['training_args']['gradient_accumulation_steps']}")
print(f"  학습률: {lora_config['training_args']['learning_rate']}")
print()

start_time = time.time()

# Actual training_hub API call
from training_hub import lora_sft

training_result = lora_sft(
    model_id=model_id,
    model_revision=model_revision,
    train_file=train_file,
    validation_file=val_file,
    output_dir=output_dir,
    lora_r=lora_config["lora"]["r"],
    lora_alpha=lora_config["lora"]["lora_alpha"],
    lora_dropout=lora_config["lora"]["lora_dropout"],
    target_modules=lora_config["lora"]["target_modules"],
    num_train_epochs=lora_config["training_args"]["num_train_epochs"],
    per_device_train_batch_size=lora_config["training_args"]["per_device_train_batch_size"],
    gradient_accumulation_steps=lora_config["training_args"]["gradient_accumulation_steps"],
    learning_rate=lora_config["training_args"]["learning_rate"],
    weight_decay=lora_config["training_args"]["weight_decay"],
    warmup_ratio=lora_config["training_args"]["warmup_ratio"],
    lr_scheduler_type=lora_config["training_args"]["lr_scheduler_type"],
    max_seq_length=lora_config["data"]["max_seq_length"],
    bf16=lora_config["training_args"]["bf16"],
    gradient_checkpointing=lora_config["training_args"]["gradient_checkpointing"],
    seed=lora_config["training"]["seed"],
    logging_steps=lora_config["training_args"]["logging_steps"],
    eval_steps=lora_config["training_args"]["eval_steps"],
    save_steps=lora_config["training_args"]["save_steps"],
    save_total_limit=lora_config["training_args"]["save_total_limit"],
    resume_from_checkpoint=lora_config["training_args"].get("resume_from_checkpoint", True),
)

wall_time = time.time() - start_time

print(f"\n✅ LoRA 학습 완료!")
print(f"  소요 시간: {wall_time/60:.1f}분")
print(f"  최종 학습 손실: {getattr(training_result, 'train_loss', 'N/A')}")
print(f"  최종 검증 손실: {getattr(training_result, 'eval_loss', 'N/A')}")

In [ ]:
"""Inspect training results — loss curve and metrics."""

import json

# Load training logs
log_history = getattr(training_result, "log_history", None)
output_dir_path = Path(output_dir)

if log_history is None:
    # Try loading from trainer_state.json
    state_path = output_dir_path / "trainer_state.json"
    if state_path.exists():
        with open(state_path) as f:
            state = json.load(f)
        log_history = state.get("log_history", [])

if log_history:
    # Extract loss values
    train_steps = [e["step"] for e in log_history if "loss" in e]
    train_losses = [e["loss"] for e in log_history if "loss" in e]
    eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
    eval_losses = [e["eval_loss"] for e in log_history if "eval_loss" in e]

    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
        ax.plot(train_steps, train_losses, label="학습 손실 (Train Loss)", alpha=0.7)
        if eval_losses:
            ax.plot(eval_steps, eval_losses, label="검증 손실 (Eval Loss)", marker="o", markersize=4)
        ax.set_xlabel("Step")
        ax.set_ylabel("Loss")
        ax.set_title("LoRA 학습 손실 곡선")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("matplotlib가 없어 텍스트로 표시합니다.")
        print("\n학습 손실 추이:")
        for step, loss in zip(train_steps[-10:], train_losses[-10:]):
            bar = "█" * int(loss * 20)
            print(f"  Step {step:>6}: {loss:.4f} {bar}")

    # Summary metrics
    print("\n--- 학습 메트릭 요약 ---")
    if train_losses:
        print(f"  초기 손실: {train_losses[0]:.4f}")
        print(f"  최종 손실: {train_losses[-1]:.4f}")
        print(f"  최소 손실: {min(train_losses):.4f}")
    if eval_losses:
        print(f"  최종 검증 손실: {eval_losses[-1]:.4f}")
        print(f"  최소 검증 손실: {min(eval_losses):.4f}")

    # GPU memory usage
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print(f"\n  피크 VRAM 사용량: {peak_vram:.1f} GB")
    print(f"  총 학습 스텝: {train_steps[-1] if train_steps else 'N/A'}")
else:
    print("학습 로그를 찾을 수 없습니다.")

In [ ]:
"""Log training results to MLflow."""

from rhoai_model_training_lab.schemas.training import TrainingResult

# Build result object
lora_result = TrainingResult(
    method="lora",
    model_id=model_id,
    model_revision=model_revision,
    bundle_id=manifest.bundle_name,
    bundle_hash=manifest.bundle_hash,
    seed=lora_config["training"]["seed"],
    train_samples=manifest.canonical_train_count,
    validation_samples=manifest.canonical_validation_count,
    total_steps=train_steps[-1] if train_steps else 0,
    final_train_loss=train_losses[-1] if train_losses else 0.0,
    final_eval_loss=eval_losses[-1] if eval_losses else None,
    best_eval_loss=min(eval_losses) if eval_losses else None,
    wall_time_seconds=wall_time,
    peak_vram_gb=peak_vram,
    gpu_name=torch.cuda.get_device_name(0),
    checkpoint_path=output_dir,
)

# Log to MLflow
mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
experiment_name = os.environ.get("MLFLOW_EXPERIMENT_TRAINING", "rhoai-model-training-lab-training")

if mlflow_uri:
    try:
        import mlflow

        mlflow.set_tracking_uri(mlflow_uri)
        mlflow.set_experiment(experiment_name)

        with mlflow.start_run(run_name=f"lora-{manifest.bundle_version}") as run:
            # Parameters
            mlflow.log_param("method", "lora")
            mlflow.log_param("model_id", model_id)
            mlflow.log_param("bundle_id", manifest.bundle_name)
            mlflow.log_param("bundle_version", manifest.bundle_version)
            mlflow.log_param("lora_r", lora_config["lora"]["r"])
            mlflow.log_param("lora_alpha", lora_config["lora"]["lora_alpha"])
            mlflow.log_param("learning_rate", lora_config["training_args"]["learning_rate"])
            mlflow.log_param("seed", lora_config["training"]["seed"])

            # Metrics
            mlflow.log_metric("final_train_loss", lora_result.final_train_loss)
            if lora_result.final_eval_loss is not None:
                mlflow.log_metric("final_eval_loss", lora_result.final_eval_loss)
            if lora_result.best_eval_loss is not None:
                mlflow.log_metric("best_eval_loss", lora_result.best_eval_loss)
            mlflow.log_metric("wall_time_seconds", lora_result.wall_time_seconds)
            mlflow.log_metric("peak_vram_gb", lora_result.peak_vram_gb)
            mlflow.log_metric("total_steps", lora_result.total_steps)
            mlflow.log_metric("train_samples", lora_result.train_samples)

            lora_result.mlflow_run_id = run.info.run_id
            lora_result.mlflow_experiment = experiment_name
            print(f"✅ MLflow 기록 완료 (run_id: {run.info.run_id})")

    except Exception as exc:
        print(f"⚠️  MLflow 기록 실패: {exc}")
        print("  학습 결과는 로컬에 저장되었습니다.")
else:
    print("⚠️  MLFLOW_TRACKING_URI 미설정 — 로컬 결과만 저장합니다.")

# Save result locally
result_path = Path(output_dir) / "training_result.json"
result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, "w") as f:
    f.write(lora_result.model_dump_json(indent=2))
print(f"📄 학습 결과 저장: {result_path}")

In [ ]:
"""Verify checkpoint and proceed to export."""

from rich.table import Table
from rich.console import Console

console = Console()

# Verify checkpoint files exist
checkpoint_dir = Path(output_dir)
adapter_config = checkpoint_dir / "adapter_config.json"
adapter_model = checkpoint_dir / "adapter_model.safetensors"

# Find the best/latest checkpoint
checkpoints = sorted(checkpoint_dir.glob("checkpoint-*"), key=lambda p: p.name)
final_adapter = checkpoint_dir  # training_hub may save directly to output_dir

checks = []

# Check adapter config
has_config = adapter_config.exists() or any(
    (cp / "adapter_config.json").exists() for cp in checkpoints
)
checks.append(("어댑터 설정 파일", has_config))

# Check adapter weights
has_weights = adapter_model.exists() or any(
    list((cp).glob("adapter_model*")) for cp in checkpoints
)
checks.append(("어댑터 가중치 파일", has_weights))

# Check tokenizer
has_tokenizer = (checkpoint_dir / "tokenizer_config.json").exists() or any(
    (cp / "tokenizer_config.json").exists() for cp in checkpoints
)
checks.append(("토크나이저 파일", has_tokenizer))

# Verify reload
reload_ok = False
try:
    from peft import PeftModel, PeftConfig

    # Try loading adapter config
    adapter_source = str(checkpoint_dir)
    if not adapter_config.exists() and checkpoints:
        adapter_source = str(checkpoints[-1])

    peft_config = PeftConfig.from_pretrained(adapter_source)
    reload_ok = True
    print(f"\n✅ 어댑터 설정 리로드 성공")
    print(f"   기본 모델: {peft_config.base_model_name_or_path}")
    print(f"   r={peft_config.r}, alpha={peft_config.lora_alpha}")
except Exception as exc:
    print(f"\n⚠️  어댑터 리로드 확인 실패: {exc}")

checks.append(("어댑터 리로드 검증", reload_ok))

# Summary table
table = Table(title="🔍 LoRA 체크포인트 검증", show_header=True)
table.add_column("항목", style="bold")
table.add_column("상태")

for name, ok in checks:
    status = "✅" if ok else "❌"
    table.add_row(name, status)

console.print(table)

all_ok = all(ok for _, ok in checks)
if all_ok:
    print("\n🎉 LoRA 학습이 성공적으로 완료되었습니다!")
    print("\n다음 단계:")
    print("  📓 04_osft_finetuning.ipynb — OSFT 학습 (독립적, 동일 데이터)")
    print("  📓 05_export_and_deploy.ipynb — 모델 내보내기 및 배포")
else:
    print("\n⚠️  일부 검증 항목이 실패했습니다. 위 결과를 확인하세요.")